
# Supply Chain Capacity Intelligence Platform

## Project Overview

This project analyzes e-commerce delivery operations and logistics performance using the Brazilian Olist dataset. The objective is to identify operational inefficiencies, SLA breach drivers, freight cost patterns, regional delivery risks, and fulfillment bottlenecks.

## Business Problem Statement

E-commerce supply chains often struggle with delayed deliveries, freight inefficiencies, operational scalability, and inconsistent SLA adherence across regions.

This analysis provides operational intelligence to support:
- Delivery reliability improvement
- Logistics optimization
- Capacity planning
- Regional operational monitoring

---

## Table of Contents

1. Import Libraries
2. Load Dataset
3. Data Overview
4. Data Cleaning & Preprocessing
5. Exploratory Data Analysis
6. Operational KPI Analysis
7. Key Business Insights
8. Business Recommendations
9. Conclusion


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Data Loading

Core datasets required for delivery operations analysis are loaded below.

In [ ]:
orders = pd.read_csv('D:/Study Items/Pathak/Data Analysis Portfolio Project/Supply Chain Capacity Intelligence Platform/Data/olist_orders_dataset.csv')

# sellers = pd.read_csv('D:/Study Items/Pathak/Data Analysis Portfolio Project/Supply Chain Capacity Intelligence Platform/Data/olist_sellers_dataset.csv')

## Initial Dataset Inspection

Objective:
Understand dataset structure, datatypes, and completeness before feature engineering.

In [ ]:
orders.head()
# customers.head()
# order_items.head()
# sellers.head()

In [ ]:
orders.info()
# customers.info()
# order_items.info()
# sellers.info()

In [ ]:
orders.isnull().sum()
# customers.info()
# order_items.info()
# sellers.info()

# Data Validation

## Objective

Validate dataset consistency and identify potential data quality issues before operational KPI engineering and delivery performance analysis.

## Duplicate Order Validation

Check whether duplicate order IDs exist in the operational dataset.

In [ ]:
duplicate_orders = orders['order_id'].duplicated().sum()

print(f'Duplicate Orders: {duplicate_orders}')

### Observation

No duplicate order IDs were identified, indicating strong primary key consistency within the orders dataset.

## Delivery Timeline Validation

Validate whether any deliveries were completed before the actual order purchase timestamp.

In [ ]:
invalid_delivery_dates = (
    orders['order_delivered_customer_date']
    < orders['order_purchase_timestamp']
).sum()

print(f'Invalid Delivery Records: {invalid_delivery_dates}')

### Observation

No invalid delivery timeline inconsistencies were identified between purchase and delivery events.

## Missing Value Analysis

Objective:
Investigate incomplete operational events and identify how missing data impacts logistics KPIs.

In [ ]:
orders.isnull().sum()

## Datetime Conversion

Convert timestamp columns into datetime format for delivery lifecycle calculations and operational KPI analysis.

In [ ]:
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)

orders['order_delivered_customer_date'] = pd.to_datetime(
    orders['order_delivered_customer_date']
)

orders['order_estimated_delivery_date'] = pd.to_datetime(
    orders['order_estimated_delivery_date']
)

## Feature Engineering

Create operational delivery metrics required for SLA monitoring and logistics performance analysis.

In [ ]:
orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date']
    - orders['order_estimated_delivery_date']
).dt.days

Positive delivery delay values indicate SLA breaches where actual delivery timelines exceeded promised customer commitments.

In [ ]:
orders['sla_breach_flag'] = (
    orders['delivery_delay_days'] > 0
).astype(int)

SLA breach tracking is critical for monitoring customer experience, operational efficiency, and logistics reliability.

In [ ]:
orders['delivery_duration_days'] = (
    orders['order_delivered_customer_date']
    - orders['order_purchase_timestamp']
).dt.days

In [ ]:
def classify_delivery(delay):

    if pd.isnull(delay):
        return 'Unknown'

    elif delay < 0:
        return 'Early'

    elif delay == 0:
        return 'On-Time'

    else:
        return 'Delayed'

In [ ]:
orders['delivery_status'] = orders[
    'delivery_delay_days'
].apply(classify_delivery)

## Delivered Orders Dataset

Create a clean analytical dataset containing only successfully delivered orders for operational KPI analysis.

In [ ]:
delivered_orders = orders[
    orders['order_delivered_customer_date'].notnull()
].copy()

In [ ]:
delivered_orders.shape

## SLA Performance Analysis

In [ ]:
sla_breach_rate = (
    delivered_orders['sla_breach_flag'].mean()
) * 100

print(round(sla_breach_rate, 2))

Approximately 6.77% of completed deliveries breached SLA timelines, indicating operational inefficiencies affecting a subset of customer shipments.

In [ ]:
delivered_orders['delivery_duration_days'].describe()

In [ ]:
delivered_orders['delivery_duration_days'].hist(
    bins=50
)

plt.title('Delivery Duration Distribution')

plt.xlabel('Delivery Duration (Days)')
plt.ylabel('Number of Deliveries')

plt.show()

## Delivery Duration Validation

Verify whether calculated delivery durations contain logically inconsistent negative values.

In [ ]:
negative_delivery_duration = (
    delivered_orders['delivery_duration_days'] < 0
).sum()

print(f'Negative Delivery Durations: {negative_delivery_duration}')

### Observation

No negative delivery durations were identified, indicating logically consistent delivery lifecycle calculations.

The delivery duration distribution is right-skewed, indicating most deliveries occur within lower-duration ranges while a smaller subset experiences significant delays.

In [ ]:
delivered_orders['delivery_status'].value_counts()

A significant majority of deliveries were completed earlier than estimated delivery timelines, suggesting strong SLA adherence or conservative delivery commitments.

# Seller Operational Performance Analysis

## Objective

Analyze seller-level delivery performance to identify operational entities contributing to delivery delays and SLA breaches.

In [ ]:
order_items = pd.read_csv('D:/Study Items/Pathak/Data Analysis Portfolio Project/Supply Chain Capacity Intelligence Platform/Data/olist_order_items_dataset.csv')

In [ ]:
seller_orders = delivered_orders.merge(
    order_items[['order_id', 'seller_id']],
    on='order_id',
    how='left'
)

In [ ]:
seller_orders.shape

In [ ]:
seller_orders[['order_id',
               'seller_id',
               'delivery_delay_days']].head()

In [ ]:
seller_delay_analysis = (
    seller_orders
    .groupby('seller_id')
    .agg({
        'delivery_delay_days': 'mean',
        'sla_breach_flag': 'mean',
        'order_id': 'count'
    })
    .reset_index()
)

In [ ]:
seller_delay_analysis.columns = [
    'seller_id',
    'avg_delay_days',
    'sla_breach_rate',
    'total_orders'
]

In [ ]:
seller_delay_analysis['sla_breach_rate'] = (
    seller_delay_analysis['sla_breach_rate'] * 100
)

In [ ]:
seller_delay_analysis.sort_values(
    by='avg_delay_days',
    ascending=False
).head(25)

In [ ]:
reliable_sellers = seller_delay_analysis[
    seller_delay_analysis['total_orders'] >= 20
]

In [ ]:
reliable_sellers.sort_values(
    by='avg_delay_days',
    ascending=False
).head(10)

# Geographic Delivery Performance Analysis

## Objective

Analyze delivery performance across customer regions to identify geographic operational bottlenecks and SLA risk areas.

In [ ]:
customers = pd.read_csv('D:/Study Items/Pathak/Data Analysis Portfolio Project/Supply Chain Capacity Intelligence Platform/Data/olist_customers_dataset.csv')


In [ ]:
geo_orders = delivered_orders.merge(
    customers[['customer_id',
               'customer_city',
               'customer_state']],
    on='customer_id',
    how='left'
)

In [ ]:
geo_orders.shape

In [ ]:
geo_orders[['customer_city',
            'customer_state',
            'delivery_delay_days']].head()

In [ ]:
state_analysis = (
    geo_orders
    .groupby('customer_state')
    .agg({
        'delivery_delay_days': 'mean',
        'sla_breach_flag': 'mean',
        'order_id': 'count'
    })
    .reset_index()
)

In [ ]:
state_analysis.columns = [
    'customer_state',
    'avg_delay_days',
    'sla_breach_rate',
    'total_orders'
]

In [ ]:
state_analysis['sla_breach_rate'] = (
    state_analysis['sla_breach_rate'] * 100
)

In [ ]:
state_analysis.sort_values(
    by='avg_delay_days',
    ascending=False
).head(10)

# Monthly Delivery Performance Trend Analysis

## Objective

Analyze delivery performance trends over time to identify seasonal operational patterns, SLA fluctuations, and potential capacity bottlenecks.

In [ ]:
delivered_orders['order_month'] = (
    delivered_orders['order_purchase_timestamp']
    .dt.to_period('M')
)

In [ ]:
monthly_analysis = (
    delivered_orders
    .groupby('order_month')
    .agg({
        'delivery_delay_days': 'mean',
        'sla_breach_flag': 'mean',
        'order_id': 'count'
    })
    .reset_index()
)

In [ ]:
monthly_analysis.columns = [
    'order_month',
    'avg_delay_days',
    'sla_breach_rate',
    'total_orders'
]

In [ ]:
monthly_analysis['sla_breach_rate'] = (
    monthly_analysis['sla_breach_rate'] * 100
)

In [ ]:
monthly_analysis['order_month'] = (
    monthly_analysis['order_month']
    .astype(str)
)

In [ ]:
monthly_analysis.head(12)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    monthly_analysis['order_month'],
    monthly_analysis['total_orders']
)

plt.xticks(rotation=90)

plt.title('Monthly Order Volume Trend')

plt.xlabel('Order Month')

plt.ylabel('Total Orders')

plt.show()

In [ ]:
monthly_analysis_filtered = monthly_analysis[
    monthly_analysis['total_orders'] >= 100
]

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    monthly_analysis_filtered['order_month'],
    monthly_analysis_filtered['total_orders']
)

plt.xticks(rotation=90)

plt.title('Monthly Order Volume Trend')

plt.xlabel('Order Month')

plt.ylabel('Total Orders')

plt.show()

## SLA Breach Trend Analysis

Analyze how SLA breach rates evolved during operational scaling and increasing shipment volumes.

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    monthly_analysis_filtered['order_month'],
    monthly_analysis_filtered['sla_breach_rate']
)

plt.xticks(rotation=90)

plt.title('Monthly SLA Breach Rate Trend')

plt.xlabel('Order Month')

plt.ylabel('SLA Breach Rate (%)')

plt.show()

In [ ]:
monthly_analysis_filtered[
    ['total_orders', 'sla_breach_rate']
].corr()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    monthly_analysis_filtered['total_orders'],
    monthly_analysis_filtered['sla_breach_rate']
)

plt.title('Order Volume vs SLA Breach Rate')

plt.xlabel('Total Orders')

plt.ylabel('SLA Breach Rate (%)')

plt.show()

In [ ]:
delivery_status_analysis = (
    delivered_orders
    .groupby('delivery_status')
    .agg({
        'delivery_duration_days': 'mean',
        'order_id': 'count'
    })
    .reset_index()
)

In [ ]:
delivery_status_analysis.columns = [
    'delivery_status',
    'avg_delivery_duration',
    'total_orders'
]

In [ ]:
delivery_status_analysis

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    delivery_status_analysis['delivery_status'],
    delivery_status_analysis['avg_delivery_duration']
)

plt.title('Average Delivery Duration by Delivery Status')

plt.xlabel('Delivery Status')

plt.ylabel('Average Delivery Duration (Days)')

plt.show()

## Geographic SLA Hotspot Analysis

Identify customer regions contributing disproportionately to SLA breach rates and operational delivery risk.

In [ ]:
state_breach_analysis = (
    geo_orders
    .groupby('customer_state')
    .agg({
        'sla_breach_flag': 'mean',
        'order_id': 'count'
    })
    .reset_index()
)

In [ ]:
state_breach_analysis.columns = [
    'customer_state',
    'sla_breach_rate',
    'total_orders'
]

In [ ]:
state_breach_analysis['sla_breach_rate'] = (
    state_breach_analysis['sla_breach_rate'] * 100
)

In [ ]:
reliable_states = state_breach_analysis[
    state_breach_analysis['total_orders'] >= 500
]

In [ ]:
reliable_states.sort_values(
    by='sla_breach_rate',
    ascending=False
).head(10)

In [ ]:
top_states = reliable_states.sort_values(
    by='sla_breach_rate',
    ascending=False
).head(10)

plt.figure(figsize=(10,5))

plt.bar(
    top_states['customer_state'],
    top_states['sla_breach_rate']
)

plt.title('Top States by SLA Breach Rate')

plt.xlabel('Customer State')

plt.ylabel('SLA Breach Rate (%)')

plt.show()

# Predictive SLA Risk Analysis

## Objective

Build a lightweight predictive model to estimate the probability of SLA breaches using operational delivery features.

In [ ]:
model_data = delivered_orders.copy()

In [ ]:
model_data['order_month_num'] = (
    model_data['order_purchase_timestamp']
    .dt.month
)

In [ ]:
X = model_data[
    ['delivery_duration_days',
     'order_month_num']
]

y = model_data['sla_breach_flag']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model = LogisticRegression()

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(y_test, y_pred))

# Executive Operational Recommendations

## Objective

Translate operational analytics findings into actionable business recommendations for improving SLA reliability, delivery efficiency, and logistics scalability.


## Key Business Insights

### Operational Findings
- SLA breach rates increased during high-volume fulfillment periods.
- Remote customer regions experienced elevated delivery risk and freight inefficiencies.
- High freight expenditure did not consistently improve delivery reliability.
- Seller concentration created operational dependency risk in specific regions.

### Key Takeaways
- Operational scalability pressure directly impacted delivery reliability.
- Freight optimization opportunities exist across high-cost regions.
- Regional fulfillment intelligence is critical for operational resilience.



## Business Recommendations

1. Expand logistics support for high-risk delivery regions.

2. Improve seasonal workforce and capacity planning during peak operational periods.

3. Optimize freight routing for regions with high cost and elevated SLA breaches.

4. Reduce operational dependency on concentrated seller hubs.

5. Implement proactive monitoring for operational bottlenecks and fulfillment risk.

### Key Takeaways
- Logistics optimization can improve delivery reliability.
- Operational planning should align with seasonal demand scaling.
- Regional diversification reduces fulfillment risk exposure.



## Conclusion

This project performed end-to-end operational analytics on e-commerce delivery performance using the Brazilian Olist dataset.

The analysis identified:
- Delivery reliability challenges
- Freight inefficiencies
- Regional SLA breach concentration
- Fulfillment scalability pressure
- Seller concentration risk

The project demonstrates capabilities in:
- Data cleaning and preprocessing
- Exploratory data analysis
- KPI engineering
- Operational analytics
- Business intelligence storytelling
- Executive-focused insight generation
